# Google public GA4 sample: descriptive findings

This notebook runs offline from committed aggregate-only artifacts retrieved from `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*` for 2020-11-01 through 2021-01-31. It contains no synthetic records or event-level data. Attribution is descriptive, not causal.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from marketing_measurement.analysis import (
    add_retention_rate,
    attribution_credits,
    funnel_rates,
)

OBSERVED = ROOT / 'data' / 'observed' / 'ga4_public_sample'
DERIVED = ROOT / 'data' / 'derived' / 'ga4_public_sample'
DERIVED.mkdir(parents=True, exist_ok=True)
metadata = json.loads((OBSERVED / 'retrieval_metadata.json').read_text())
funnel = pd.read_json(OBSERVED / 'funnel_daily_by_channel.json')
cohorts = pd.read_json(OBSERVED / 'cohort_retention.json')
paths = pd.read_json(OBSERVED / 'conversion_channel_paths.json')

funnel['event_date'] = pd.to_datetime(funnel['event_date'])
assert metadata['project_id'] == 'christina-data-portfolio-2026'
assert metadata['maximum_bytes_billed'] == 4_000_000_000
assert funnel['event_date'].min() == pd.Timestamp('2020-11-01')
assert funnel['event_date'].max() == pd.Timestamp('2021-01-31')
assert {'user_pseudo_id', 'ga_session_id', 'transaction_id'}.isdisjoint(paths.columns)


In [2]:
funnel_totals = funnel[['views', 'engaged_sessions', 'add_to_carts', 'checkouts', 'purchases']].sum().to_frame().T
funnel_summary = funnel_rates(funnel_totals).iloc[0]

cohorts['cohort_date'] = pd.to_datetime(cohorts['cohort_date'])
cohorts = add_retention_rate(cohorts)
cohort_sizes = cohorts.loc[cohorts['days_since_acquisition'].eq(0)].set_index('cohort_date')['cohort_users']
day_seven = cohorts.loc[cohorts['days_since_acquisition'].eq(7)].set_index('cohort_date')['retained_users']
eligible_cohorts = cohort_sizes.index <= pd.Timestamp('2021-01-24')
day_seven = day_seven.reindex(cohort_sizes.index, fill_value=0)
eligible_users = int(cohort_sizes[eligible_cohorts].sum())
day_seven_users = int(day_seven[eligible_cohorts].sum())

path_credit_rows = []
for path_index, path in paths.iterrows():
    channels = str(path['channel_path']).split(' > ')
    path_credit_rows.extend(
        {'conversion_id': f'observed_path_{path_index}', 'channel': channel, 'touch_number': touch, 'conversion_weight': int(path['converted_sessions'])}
        for touch, channel in enumerate(channels, start=1)
    )
path_frame = pd.DataFrame(path_credit_rows)
attribution_summary = {}
for model in ('first_touch', 'last_touch', 'linear', 'time_decay'):
    credits = attribution_credits(path_frame, model)
    credits['weighted_credit'] = credits['credit'] * credits['conversion_weight']
    attribution_summary[model] = (
        credits.groupby('channel')['weighted_credit'].sum().sort_values(ascending=False).head(8).round(6).to_dict()
    )

summary = {
    'source': {
        'dataset': 'bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*',
        'coverage_start': '2020-11-01',
        'coverage_end': '2021-01-31',
        'retrieved_at_utc': metadata['retrieved_at_utc'],
    },
    'funnel': {
        'views': int(funnel_summary['views']),
        'engaged_sessions': int(funnel_summary['engaged_sessions']),
        'add_to_carts': int(funnel_summary['add_to_carts']),
        'checkouts': int(funnel_summary['checkouts']),
        'purchases': int(funnel_summary['purchases']),
        'engagement_rate': round(float(funnel_summary['engaged_sessions_rate']), 6),
        'add_to_cart_rate': round(float(funnel_summary['add_to_carts_rate']), 6),
        'checkout_rate': round(float(funnel_summary['checkouts_rate']), 6),
        'purchase_rate': round(float(funnel_summary['purchases_rate']), 6),
    },
    'day_7_retention': {
        'eligible_cohorts': int(eligible_cohorts.sum()),
        'cohort_users': eligible_users,
        'retained_users': day_seven_users,
        'rate': round(day_seven_users / eligible_users, 6),
    },
    'attribution': {
        'converted_sessions': int(paths['converted_sessions'].sum()),
        'interpretation': 'descriptive attribution; not causal',
        'top_channel_credits': attribution_summary,
    },
}
(DERIVED / 'findings_summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True) + '\n')
summary


{'source': {'dataset': 'bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*',
  'coverage_start': '2020-11-01',
  'coverage_end': '2021-01-31',
  'retrieved_at_utc': '2026-09-11T13:49:05.887878+00:00'},
 'funnel': {'views': 334282,
  'engaged_sessions': 250575,
  'add_to_carts': 14953,
  'checkouts': 5945,
  'purchases': 2834,
  'engagement_rate': 0.749592,
  'add_to_cart_rate': 0.059675,
  'checkout_rate': 0.397579,
  'purchase_rate': 0.476703},
 'day_7_retention': {'eligible_cohorts': 85,
  'cohort_users': 250712,
  'retained_users': 1966,
  'rate': 0.007842},
 'attribution': {'converted_sessions': 4848,
  'interpretation': 'descriptive attribution; not causal',
  'top_channel_credits': {'first_touch': {'google / organic': 1555.0,
    '(direct) / (none)': 1104.0,
    '<Other> / <Other>': 707.0,
    '<Other> / referral': 446.0,
    'shop.googlemerchandisestore.com / referral': 384.0,
    '(data deleted) / (data deleted)': 273.0,
    'google / cpc': 228.0,
    '<Other> / orga

## Findings

1. In this Google sample, 250,575 of 334,282 viewed sessions were engaged (74.96%); 14,953 engaged sessions reached cart (5.97%), 5,945 reached checkout (39.76%), and 2,834 reached purchase (47.67%). Source: `funnel_daily_by_channel.sql` and `findings_summary.json`.
2. Across 85 acquisition cohorts with a fully observable seventh day, 1,966 of 250,712 cohort users returned on day 7 (0.78%). Source: `cohort_retention.sql` and `findings_summary.json`.
3. For 4,848 converted sessions, Google / organic received 1,555 first-touch credits and 1,250 last-touch credits; this is descriptive attribution, not causal. Source: `conversion_channel_paths.sql` and `findings_summary.json`.

The public sample is obfuscated, user-acquisition source/medium is used as the channel label, and incomplete future follow-up excludes the most recent seven acquisition dates from the day-7 retention denominator.